In [1]:
# import pandas as pd
# file_df = pd.read_csv("Greenway_Serono_phase2_PHI_Analysis_1.csv")
# filtered_df = file_df.loc[file_df['TABLE_NAME'] == 'ERXPatMedList']

# filtered_df.to_csv("filtered_df.csv", index=False)

In [1]:
import requests
import pandas as pd

url = "http://localhost:8000/upload_config_from_csv/1/"

# input_csv = "phi_pr2.csv"
# input_csv = "phi_pr1_p1.csv"
# input_csv = "phi_ad_mci.csv"
# input_csv = "phi_custom.csv"
input_csv = "phi_clinicalbin.csv"

files = {"file": ("all_tables.csv", open(input_csv, "rb"), "text/csv")}

response = requests.post(url, files=files)

print(response.status_code, response.json())

200 {'success': True, 'message': 'Configuration uploaded successfully.'}


In [2]:
df = pd.read_csv(input_csv)
tables = df['TABLE_NAME'].unique()
len(tables)

1

In [3]:
notes_table = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'NOTES']['TABLE_NAME'].to_list()))
len(notes_table)

1

In [4]:
no_notes_df = df[~df['TABLE_NAME'].isin(notes_table)]
no_notes_table = list(set(no_notes_df['TABLE_NAME'].to_list()))
len(no_notes_table)

0

In [5]:
static_df = no_notes_df[no_notes_df['DE_IDENTIFICATION_RULE'] == 'STATIC_OFFSET']
static_tables = list(set(static_df['TABLE_NAME'].to_list()))
len(static_tables)

0

In [6]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/ndai/Desktop/deidentification/deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

import pandas as pd
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel, IgnoreRowsDeIdentificaiton
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task, _get_default_table_details_for_ui

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [7]:
def fix_reference_value(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "PATIENT_ID"
    ]
    enc_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "ENCOUNTER_ID"
    ]
    print(pid_col)
    print(enc_col)
    patient_id = pid_col[0] if len(pid_col)>0 else None
    enc_id = enc_col[0] if len(enc_col)>0 else None
    table.table_details_for_ui['reference_enc_id_column'] = enc_id
    table.table_details_for_ui['reference_patient_id_column'] = patient_id
    table.save()

In [8]:
db, _ = DbDetailsModel.objects.get_or_create(db_name="deid_prod")
db.id

1

In [66]:
to = TableDetailsModel.objects.get(table_name="OrdReqOrder")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'DocumentID',
    'source_column': 'DocumentID',
    'reference_table': 'ClinicalDocuments'}],
  'source_table': 'OrdReqOrder',
  'destination_column': 'PatientID',
  'destination_column_type': 'patient_id'}

to.save()

In [70]:
# to = TableDetailsModel.objects.get(table_name="OrdReqDiagnosis")
# to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'OrdReqID',
#     'source_column': 'OrdReqID',
#     'reference_table': 'OrdReqClinicalDocuments'},
#                    {'column_name': 'DocumentID',
#     'source_column': 'DocumentID',
#     'reference_table': 'ClinicalDocuments'}],
#   'source_table': 'OrdReqDiagnosis',
#   'destination_column': 'PatientID',
#   'destination_column_type': 'patient_id'}

# to.save()

In [71]:
to = TableDetailsModel.objects.get(table_name="OrdReqNote")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'OrdReqID',
    'source_column': 'OrdReqID',
    'reference_table': 'OrdReqClinicalDocuments'},
                   {'column_name': 'DocumentID',
    'source_column': 'DocumentID',
    'reference_table': 'ClinicalDocuments'}],
  'source_table': 'OrdReqNote',
  'destination_column': 'PatientID',
  'destination_column_type': 'patient_id'}

to.save()

In [13]:
to = TableDetailsModel.objects.get(table_name="OBXNote")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'OBXId',
    'source_column': 'OBXId',
    'reference_table': 'OBX'}],
  'source_table': 'OBXNote',
  'destination_column': 'PatientID',
  'destination_column_type': 'patient_id'}

to.save()

In [73]:
# to = TableDetailsModel.objects.get(table_name="ClinicalPatMedsFillStatus")
# to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ClinicalPatMedID',
#     'source_column': 'ClinicalPatMedID',
#     'reference_table': 'ClinicalPatMeds'}],
#   'source_table': 'ClinicalPatMedsFillStatus',
#   'destination_column': 'PatientID',
#   'destination_column_type': 'patient_id'}

# to.save()

In [15]:
# to = TableDetailsModel.objects.get(table_name="ClinicalBin_XML")
# to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'DocumentID',
#     'source_column': 'DocumentID',
#     'reference_table': 'ClinicalDocuments'}],
#   'source_table': 'ClinicalBin_XML',
#   'destination_column': 'PatientID',
#   'destination_column_type': 'patient_id'}

# to.save()

In [9]:
TableDetailsModel.objects.filter(table_status=0, db=db.id).count()

8

In [10]:
TableDetailsModel.objects.filter(table_status=1, db=db.id).count()

4

In [11]:
TableDetailsModel.objects.filter(table_status=2, db=db.id).count()

190

In [12]:
TableDetailsModel.objects.filter(table_status=3, db=db.id).count()

0

In [45]:
Task.objects.filter(status=0).count()

395

In [46]:
Task.objects.filter(status=1).count()

0

In [43]:
Task.objects.filter(status=2).count()

245

In [44]:
Task.objects.filter(status=-1).count()

132

In [21]:
# Task.objects.filter(status=0).update(status=-1)
# Task.objects.filter(status=0).update(failure_count=0)

In [22]:
# Chain.objects.all().delete()

In [23]:
from django.conf import settings
settings.BATCH_SIZE_DURING_DE_IDENTIFICATION, settings.DEFAULT_OFFSET_VALUE
# settings.DB_CACHER

(10000, 30)

In [27]:
# # df = pd.read_csv("p1_b3_tables.csv")
# # tables = df['table_name'].to_list()
# # tables = ['ERXPatMedList']
# tables = ["ClinicalDocuments_LillyAD"]
# tables = [table_obj.table_name for table_obj in TableDetailsModel.objects.filter(table_status=0, db=db.id)]
# len(tables)

In [24]:
# tables = ['Visit']
completed = []
failed_tables = []
len(tables), len(completed), len(failed_tables)

(1, 0, 0)

In [30]:
# tables = ['ClinicalBin_decrypt']

In [25]:
#to run table from a specific DB model
_db = DbDetailsModel.objects.get(db_name = "deid_prod")
totalrun = 0
for table in tables:
    print(table)
    tableobj = TableDetailsModel.objects.get(table_name=table, db=_db)
    Task.objects.filter(arguments__table_id=tableobj.id).delete()
    # if tableobj.table_status in [2]:
    #     continue
    fix_reference_value(tableobj.id)
    tableobj.refresh_from_db()
    tasks, chain = create_deidentification_task(tableobj, True)
    totalrun += 1
    print(tableobj.rows_count)

2026-02-17 10:33:03,231 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2026-02-17 10:33:03,240 - deIdentification.nd_logger - WARNING - Table clinicalbin_decrypted does not exist and cannot be dropped.
2026-02-17 10:33:03,241 - deIdentification.nd_logger - INFO - Dropping ignore rows for clinicalbin_decrypted, deid_prod


clinicalbin_decrypted
['patient_id']
['visit_id']
4457861
